In [ ]:
#@title Install OmniVoice

%cd /content/

# Clean old files
!rm -rf omnivoice-colab
!rm -rf OmniVoice

# Clone YOUR repo
!git clone https://github.com/hahyt6644-gif/omnivoice-colab.git

%cd /content/omnivoice-colab

# Clone OFFICIAL OmniVoice repo only
!git clone https://github.com/k2-fsa/OmniVoice.git

# Install ffmpeg
!apt-get update -qq
!apt-get install -y ffmpeg

# Install requirements
!pip install -q -r colab.txt

# Extra packages needed for API
!pip install -q \
fastapi \
"uvicorn[standard]" \
python-multipart \
httpx \
requests \
aiofiles \
nest-asyncio \
python-dotenv \
pydub \
soundfile \
librosa

# Download latest app.py from YOUR repo
!wget -q -O app.py https://raw.githubusercontent.com/hahyt6644-gif/omnivoice-colab/main/app.py

# GPU check
import torch

print("========== SYSTEM INFO ==========")

if torch.cuda.is_available():
    print("✅ GPU ENABLED")
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "❌ GPU NOT ENABLED!\n"
        "Runtime → Change runtime type → GPU"
    )

from IPython.display import clear_output
clear_output()

print("✅ OmniVoice Installed Successfully")

In [ ]:
#@title Run OmniVoice APP + Cloudflare URL

%cd /content/omnivoice-colab

# Kill previous server
!pkill -9 -f "python app.py" || true
!pkill -9 -f "cloudflared" || true
!pkill -9 -f "uvicorn" || true

# Install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb >/dev/null 2>&1

import subprocess
import threading
import time
import re
from IPython.display import clear_output

print("🚀 Starting OmniVoice...")

# Start app.py
server = subprocess.Popen(
    ["python", "app.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Show logs
def logs():
    while True:
        line = server.stdout.readline()
        if not line:
            break
        print(line.strip())

threading.Thread(target=logs, daemon=True).start()

# Wait for model loading
time.sleep(25)

print("\n🌍 Starting Cloudflare Tunnel...\n")

# Start tunnel
tunnel = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--url",
        "http://localhost:7860",
        "--no-autoupdate"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

url = None

for _ in range(120):
    line = tunnel.stdout.readline()

    if line:
        match = re.search(
            r'https://[-a-zA-Z0-9]+\.trycloudflare\.com',
            line
        )

        if match:
            url = match.group(0)
            break

    time.sleep(1)

clear_output()

if url:
    print("=" * 60)
    print("✅ OMNIVOICE RUNNING")
    print("=" * 60)

    print("\n🌍 Public URL:")
    print(url)

    print("\n🖥 UI:")
    print(url + "/ui")

    print("\n🎙 Clone API:")
    print(url + "/api/clone")

    print("\n🎨 Design API:")
    print(url + "/api/design")

    print("\n⚠️ Keep notebook running")
    print("⚠️ Restart = New URL")
    print("=" * 60)

else:
    print("❌ Failed to get Cloudflare URL")